In [2]:
from datetime import time
import random
from dotenv import load_dotenv
import lyricsgenius
import os


env_path = os.path.join(os.path.dirname(
    os.path.dirname(os.path.abspath('__file__'))), '.env')
load_dotenv(env_path, override=True, encoding='utf-8')

# 1) Genius API Token
GENIUS_API_TOKEN = os.getenv("GENIUS_API_TOKEN")
if not GENIUS_API_TOKEN:
    raise ValueError("GENIUS_API_TOKEN not found in .env file")

# 2) Path to your artist list (one artist name per line)
ARTIST_LIST_PATH = "another_artists_list_300.txt"

# 3) Output CSV (where we'll append results as we go)
OUTPUT_CSV = "scraped_lyrics_2.csv"


# 4) How many songs to fetch per artist
SONGS_PER_ARTIST = int(os.getenv("SONGS_PER_ARTIST", "25"))


# 5) Pause (seconds) between artist requests to avoid rate-limiting
SLEEP_BETWEEN_ARTISTS = float(os.getenv("SLEEP_BETWEEN_ARTISTS", "1.5"))


# 6) Rate limit handling configuration
INITIAL_BACKOFF = int(os.getenv("INITIAL_BACKOFF", 10)
                      )  # Start with 10 seconds
MAX_RETRIES = int(os.getenv("MAX_RETRIES", 5))       # Try up to 5 times

GENIUS_API_TOKEN = os.getenv("GENIUS_API_TOKEN")
load_dotenv(override=True)

genius = lyricsgenius.Genius(
    GENIUS_API_TOKEN,
    timeout=15,
    retries=3,
    sleep_time=0.25,  # small pause between each page scrape
    # Exclude these terms from song titles
    excluded_terms=["(Remix)", "(Live)"],
    skip_non_songs=True,  # Skip non-song entries (e.g., interviews)
    # Remove section headers like "Verse", "Chorus"
)


def with_rate_limit_handling(api_function):
    """Decorator to handle rate limit errors with exponential backoff"""
    def wrapper(*args, **kwargs):
        for attempt in range(MAX_RETRIES + 1):
            try:
                return api_function(*args, **kwargs)
            except Exception as e:
                error_str = str(e)
                # Check if it's a rate limit error
                if "429" in error_str and attempt < MAX_RETRIES:
                    # Calculate backoff time with jitter
                    backoff_time = INITIAL_BACKOFF * \
                        (2 ** attempt) + random.uniform(1, 5)
                    print(
                        f"\nRate limit exceeded. Waiting {backoff_time:.1f} seconds before retry {attempt+1}/{MAX_RETRIES}")
                    time.sleep(backoff_time)
                else:
                    if "429" in error_str:
                        print(
                            f"\nRate limit exceeded after {MAX_RETRIES} retries. Consider increasing wait time.")
                    raise
    return wrapper


def fetch_artist_lyrics(artist_name, max_songs=SONGS_PER_ARTIST):
    """
    Fetch up to max_songs tracks for `artist_name`, returning a list of dicts
    """
    songs_data = []
    try:
        # Search for the artist with rate limit handling
        artist_obj = search_artist(artist_name, max_songs)

        if artist_obj is None or not artist_obj.songs:
            print(f"  → No songs found for artist: {artist_name}")
            return songs_data

        for song in artist_obj.songs:
            title = song.title.strip()
            lyrics = song.lyrics.strip()

            # Skip extremely short lyrics (e.g., < 20 chars)
            if len(lyrics) < 20:
                continue
            songs_data.append({
                "artist": artist_name,
                "song_title": title,
                "lyrics": lyrics
            })

    except Exception as e:
        print(f"ERROR: Could not search for artist [{artist_name}]: {e}")

    return songs_data
# -----------------------------
# HELPER FUNCTION: fetch_artist_lyrics
# -----------------------------
@with_rate_limit_handling
def search_artist(artist_name, max_songs):
    """Search for an artist with rate limit handling"""
    return genius.search_artist(artist_name, max_songs=max_songs, sort="popularity", get_full_info=False)


@with_rate_limit_handling
def search_song(title, artist):
    """Search for a song with rate limit handling"""
    return genius.search_song(title=title, artist=artist, get_full_info=False)


song = search_song("I'm Yours", "Jason Mraz"
                   )

Searching for "I'm Yours" by Jason Mraz...
Done.


d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:468: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


In [4]:
artist = search_artist('Ali Farka Touré', max_songs=10)

Searching for songs by Ali Farka Touré...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "Ai Du"
Song 2: "Diaraby"
Song 3: "Erdi"
Song 4: "Ruby"
Song 5: "Beto"
"Tamalla" is not valid. Skipping.
Song 6: "Mali dje"
"Cherie" is not valid. Skipping.
"Ali Hala Abada" is not valid. Skipping.
"Instrumental" is not valid. Skipping.
"Safari" is not valid. Skipping.
Done. Found 6 songs.


In [5]:
print(artist.songs)

[Song(id, artist, ...), Song(id, artist, ...), Song(id, artist, ...), Song(id, artist, ...), Song(id, artist, ...), Song(id, artist, ...)]


In [2]:
import pandas as pd
import re

data_path = "./Lyrics_extraction/scraped_lyrics_no_metadata_combined.csv"
df = pd.read_csv(data_path)


def clean_lyrics_for_ctm(text):
    # Remove structural markers like [Chorus], [Verse], etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Clean up whitespace and newlines
    text = re.sub(r'\n+', ' ', text)  # Replace newlines with spaces
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace

    return text.strip()


df['lyrics'] = df['lyrics'].apply(clean_lyrics_for_ctm)

print(f"📊 Dataset Overview:")
print(f"   Songs: {len(df):,}")
print(f"   Artists: {df['artist'].nunique():,}")
print(f"   Columns: {list(df.columns)}")

# Display sample data
print("\n🎵 Sample songs:")
df.head(3)

📊 Dataset Overview:
   Songs: 5,971
   Artists: 598
   Columns: ['artist', 'song_title', 'lyrics']

🎵 Sample songs:


,artist,song_title,lyrics
0,21 Savage,Bank Account,"Ooh, ooh, oh, oh, oh, ow, ow Wow, wow, wow, ah..."
1,21 Savage,Ghostface Killers,"Metro Boomin want some more, nigga (Hey) Autom..."
2,21 Savage,Glock in My Lap,"Y'all niggas stop playin', nigga Y'all niggas ..."
